# 📱 All-in-One Mobile Cloud Engine for AI Video Studio APK
Kyunki Android phone par heavy Python AI libraries aur FFmpeg directly nahi chal saktin, yeh Colab notebook aapki Android App ke liye **Free Cloud GPU Engine** ka kaam karti hai:
1. Menu mein **Runtime -> Change runtime type -> T4 GPU** select karein.
2. **Runtime -> Run all** dabayein.
3. Neeche jo `https://xxxx.trycloudflare.com` URL print ho, usay apni **AI Video Studio Android App** mein paste kar dein!

In [ ]:
# Step 1: Install Studio Libraries + GPU Lip-Sync + Tunnel
!pip install -q streamlit google-genai edge-tts moviepy==1.0.3 pillow requests nest-asyncio
!pip install -q --upgrade google-genai
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
!git clone -q https://github.com/Rudrabha/Wav2Lip.git
!pip install -q librosa==0.9.2 opencv-python-headless scipy tqdm numba
!wget -q 'https://iiitaphyd-my.sharepoint.com/personal/radrabha_m_research_iiit_ac_in/_layouts/15/download.aspx?share=EdjI7bZlgApMqsVoEUUXpLsBxqXbn5z8VTmoxp55YNDcIA' -O Wav2Lip/checkpoints/wav2lip_gan.pth
print('✅ Mobile Cloud Engine & T4 GPU Lip-Sync Ready!')

In [ ]:
# Step 2: Write Mobile-Optimized Streamlit Dashboard
%%writefile mobile_studio.py
import os, json, time, asyncio, subprocess
import streamlit as st
import edge_tts
from PIL import Image
from google import genai
from google.genai import types
from moviepy.editor import VideoFileClip, AudioFileClip, CompositeAudioClip, concatenate_videoclips, vfx

st.set_page_config(page_title='AI Video Studio Mobile', layout='wide')
for d in ['work/refs', 'work/images', 'work/audio', 'work/raw', 'work/synced']:
    os.makedirs(d, exist_ok=True)

if 'scenes' not in st.session_state: st.session_state.scenes = []

st.markdown('### 📱 Mobile Studio Controls')
gemini_key = st.text_input('Free Gemini API Key', type='password')
c1, c2 = st.columns(2)
with c1:
    host_img = st.file_uploader('Host Photo', type=['png','jpg','jpeg'])
    if host_img: open('work/refs/host.png','wb').write(host_img.read())
with c2:
    guest_img = st.file_uploader('AI Guest Photo', type=['png','jpg','jpeg'])
    if guest_img: open('work/refs/guest.png','wb').write(guest_img.read())

aspect = st.selectbox('Aspect Ratio', ['9:16', '16:9'])
num_scenes = st.slider('Scenes', 2, 8, 4)
topic = st.text_input('Video Topic', placeholder='e.g. AI aur insaanon ka future...')

if st.button('1️⃣ Generate Script & Nano Banana Frames', type='primary', use_container_width=True):
    client = genai.Client(api_key=gemini_key)
    prompt = f'Create a {num_scenes}-scene viral podcast & VFX storyboard on "{topic}". Return JSON with "scenes" list: scene_id, speaker (Host/AI_Avatar/None), dialogue_roman_urdu (12 words max), image_prompt ({aspect}), veo_motion_prompt, needs_lipsync (bool).'
    res = client.models.generate_content(model='gemini-3.8-flash', contents=prompt, config=types.GenerateContentConfig(response_mime_type='application/json'))
    st.session_state.scenes = json.loads(res.text)['scenes']
    for sc in st.session_state.scenes:
        sid = sc['scene_id']
        ref_p = 'work/refs/host.png' if sc['speaker']=='Host' else ('work/refs/guest.png' if sc['speaker']=='AI_Avatar' else None)
        contents = [Image.open(ref_p), f'Keep exact identity from reference. {aspect} cinematic frame: {sc["image_prompt"]}'] if (ref_p and os.path.exists(ref_p)) else [sc['image_prompt']]
        img_res = client.models.generate_content(model='gemini-3.8-flash', contents=contents, config=types.GenerateContentConfig(response_modalities=['IMAGE']))
        for part in img_res.candidates[0].content.parts:
            if part.inline_data: open(f'work/images/s_{sid}.png','wb').write(part.inline_data.data)
    st.success('Script & Consistent Frames Ready!')

for sc in st.session_state.scenes:
    sid = sc['scene_id']
    with st.expander(f'Scene {sid} ({sc["speaker"]})', expanded=True):
        if os.path.exists(f'work/images/s_{sid}.png'):
            st.image(f'work/images/s_{sid}.png', width=200)
            st.download_button(f'⬇️ Download Frame #{sid}', open(f'work/images/s_{sid}.png','rb'), file_name=f'frame_{sid}.png', key=f'df_{sid}')
        st.code(sc['veo_motion_prompt'])
        up = st.file_uploader(f'Upload Veo Clip #{sid} (from Google Flow button above)', type=['mp4'], key=f'uv_{sid}')
        if up: open(f'work/raw/s_{sid}.mp4','wb').write(up.read()); st.success('Clip Saved!')

if st.button('2️⃣ Generate Urdu Voices + GPU Lip-Sync + Final MP4', type='primary', use_container_width=True):
    clips = []
    for sc in st.session_state.scenes:
        sid = sc['scene_id']
        raw_p = f'work/raw/s_{sid}.mp4'
        if not os.path.exists(raw_p): continue
        dlg = sc.get('dialogue_roman_urdu','').strip()
        if dlg:
            voice = 'ur-PK-AsadNeural' if sc['speaker']=='Host' else 'ur-PK-UzmaNeural'
            aud_p = f'work/audio/s_{sid}.mp3'
            wav_p = f'work/audio/s_{sid}.wav'
            asyncio.run(edge_tts.Communicate(dlg, voice).save(aud_p))
            subprocess.run(['ffmpeg','-y','-i',aud_p,'-ar','16000','-ac','1',wav_p], check=True)
            if sc.get('needs_lipsync'):
                sync_p = f'work/synced/s_{sid}.mp4'
                subprocess.run(['python','Wav2Lip/inference.py','--checkpoint_path','Wav2Lip/checkpoints/wav2lip_gan.pth','--face',raw_p,'--audio',wav_p,'--outfile',sync_p], check=True)
                clips.append(VideoFileClip(sync_p))
            else:
                c = VideoFileClip(raw_p)
                a = AudioFileClip(aud_p)
                clips.append(c.subclip(0, min(c.duration, a.duration)).set_audio(a))
        else:
            clips.append(VideoFileClip(raw_p))
    if clips:
        final = concatenate_videoclips(clips, method='compose')
        final.write_videofile('work/final_mobile_video.mp4', codec='libx264', audio_codec='aac', fps=24, logger=None)
        st.video('work/final_mobile_video.mp4')
        st.download_button('⬇️ Download Final Video to Phone', open('work/final_mobile_video.mp4','rb'), file_name='ai_studio_final.mp4', type='primary')


In [ ]:
# Step 3: Launch Streamlit + Cloudflared Public Tunnel for Android APK
import subprocess, threading, re, time

subprocess.Popen(['streamlit', 'run', 'mobile_studio.py', '--server.port', '8501', '--server.headless', 'true'])
time.sleep(3)

proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8501'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('\n📱 COPY THIS URL INTO YOUR ANDROID APP (AI Video Studio):')
        print(m.group(0))
        break
